In [149]:
import pandas as pd

path = "../../output/tnm/mechanical_switch_final.xlsx" 
xls = pd.ExcelFile(path)
print("Available sheets:", xls.sheet_names)


Available sheets: ['Sheet', 'hitachi', 'srb', 'mhe-demag', 'mechanical-switch']


In [150]:
selected_sheets = xls.sheet_names

dfs = {s: xls.parse(s) for s in selected_sheets}

for name, df in dfs.items():
    if name == "Sheet":
        continue
    print(f"{name}: shape={df.shape}")
    # print(df.head())

hitachi: shape=(162, 91)
srb: shape=(44, 67)
mhe-demag: shape=(87, 50)
mechanical-switch: shape=(42, 67)


In [151]:
final_json = {}
mechswitch_reference = {}

In [152]:
bool_errors = []

def normalize_mechswitch_key(col):
    """
    Convert a completed column to standard .status key.
    Example: 'switch_beam.a.completed?' -> 'switch_beam.a.status'
    """
    if not col.endswith(".completed?") and not col.endswith(".completed"):
        return None

    # remove suffix
    base = col.replace(".completed?", "").replace(".completed", "")
    return f"{base}.status"


def to_bool(value, workorder_no=None, filename=None, column=None):
    """
    Convert various values to boolean for status columns.
    Unknown formats are stored as False and logged.
    """
    try:
        if pd.isna(value):
            return False

        if isinstance(value, bool):
            return value

        if isinstance(value, (int, float)):
            return value == 1

        if isinstance(value, str):
            v = value.strip().lower()
            if v in ["yes", "y", "true", "1", "checked", "ok", "✓"]:
                return True
            if v in ["no", "n", "false", "0", "unchecked", "x", "✗"]:
                return False

        # Unknown format
        if workorder_no and filename and column:
            bool_errors.append({
                "workorder_no": workorder_no,
                "filename": filename,
                "column": column,
                "value": value,
                "error": "Unknown boolean format, storing as False"
            })
        return False

    except Exception as e:
        if workorder_no and filename and column:
            bool_errors.append({
                "workorder_no": workorder_no,
                "filename": filename,
                "column": column,
                "value": value,
                "error": str(e)
            })
        return False


def get_inspection_item_col(completed_col):
    """
    Return the corresponding inspection_item column for a completed column
    """
    return completed_col.replace(".completed?", ".inspection_item").replace(".completed", ".inspection_item")


def process_mechswitch_sheet(df, brand):
    """
    Process mechanical switch sheet and build:
    - final_json: {workorder_no: {"filename": ..., "data": {...}}}
    - mechswitch_reference: {status_key: {"inspection_item": ..., "brands": set()}}
    """
    for _, row in df.iterrows():
        workorder_no = row["workorder_id"]
        filename = row["filename"]

        data = {}

        for col in df.columns:
            if col in ["workorder_id", "filename"]:
                continue

            # -----------------------------
            # 1️⃣ Status columns
            # -----------------------------
            if col.endswith(".completed?") or col.endswith(".completed"):
                base_key = normalize_mechswitch_key(col)
                if base_key is None:
                    continue

                # status key already ends with .status
                status_key = base_key
                value = row[col]
                bool_value = to_bool(value, workorder_no, filename, col)
                data[status_key] = bool_value

                # ---- reference extraction ----
                inspection_col = get_inspection_item_col(col)
                if inspection_col in df.columns:
                    inspection_value = row[inspection_col]
                    if pd.notna(inspection_value):
                        if base_key not in mechswitch_reference:
                            mechswitch_reference[base_key] = {
                                "inspection_item": str(inspection_value),
                                "brands": {brand}
                            }
                        else:
                            mechswitch_reference[base_key]["brands"].add(brand)

                continue  # move to next column

            # -----------------------------
            # 2️⃣ Metadata / other columns
            # -----------------------------
            # copy any non-status columns automatically
            data[col] = row[col]

        # -----------------------------
        # 3️⃣ Merge into final_json
        # -----------------------------
        if workorder_no not in final_json:
            final_json[workorder_no] = {"filename": filename, "data": data}
        else:
            final_json[workorder_no]["data"].update(data)
            if not final_json[workorder_no].get("filename"):
                final_json[workorder_no]["filename"] = filename


### Hitachi

In [153]:
df_mechswitch_hitachi = dfs.get("hitachi")

for _, row in df_mechswitch_hitachi.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_mechswitch_hitachi.columns:
        if col in ["workorder_id", "filename"]:
            continue

        new_key = normalize_mechswitch_key(col)
        if new_key is not None:
            value = row[col]
            data[new_key] = to_bool(value, workorder_no, filename, col)
            continue

        data[col] = row[col]

    if workorder_no not in final_json:
        final_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        final_json[workorder_no]["data"].update(data)

        if not final_json[workorder_no].get("filename"):
            final_json[workorder_no]["filename"] = filename


In [154]:
final_json

{4000484591: {'filename': 'TN_PM_MTH_MechanicalSwitch1_4000484591.pdf',
  'data': {'non_rumba_mechanism.e.type': "Visual and tightening torques (Cont'd)",
   'non_rumba_mechanism.e.inspection_item': 'Non-Rumba plates secured tightly at each end.',
   'non_rumba_mechanism.e.status': True,
   'non_rumba_mechanism.f.type': "Visual and tightening torques (Cont'd)",
   'non_rumba_mechanism.f.inspection_item': 'Non-Rumba cams firmly secured.',
   'non_rumba_mechanism.f.status': True,
   'non_rumba_mechanism.g.type': "Visual and tightening torques (Cont'd)",
   'non_rumba_mechanism.g.inspection_item': 'Remove manhole cover and inspect internally the non-rumba operating mechanism.',
   'non_rumba_mechanism.g.status': True,
   'switch_beam.a.type': 'Visual and tightening torques',
   'switch_beam.a.inspection_item': 'Top and side finger plates secured.',
   'switch_beam.a.status': True,
   'switch_beam.b.type': 'Visual and tightening torques',
   'switch_beam.b.inspection_item': 'Beam and carri

### MHE-DEMAG

In [155]:
df_mechswitch_mhedemag = dfs.get("mhe-demag")

for _, row in df_mechswitch_mhedemag.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_mechswitch_mhedemag.columns:
        if col in ["workorder_id", "filename"]:
            continue

        new_key = normalize_mechswitch_key(col)
        if new_key is not None:
            value = row[col]
            data[new_key] = to_bool(value, workorder_no, filename, col)
            continue

        data[col] = row[col]

    if workorder_no not in final_json:
        final_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        final_json[workorder_no]["data"].update(data)

        if not final_json[workorder_no].get("filename"):
            final_json[workorder_no]["filename"] = filename


### SRB

In [156]:
df_mechswitch_srb = dfs.get("srb")

for _, row in df_mechswitch_srb.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_mechswitch_srb.columns:
        if col in ["workorder_id", "filename"]:
            continue

        new_key = normalize_mechswitch_key(col)
        if new_key is not None:
            value = row[col]
            data[new_key] = to_bool(value, workorder_no, filename, col)
            continue

        data[col] = row[col]

    if workorder_no not in final_json:
        final_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        final_json[workorder_no]["data"].update(data)

        if not final_json[workorder_no].get("filename"):
            final_json[workorder_no]["filename"] = filename


### Normal

In [157]:
df_mechswitch = dfs.get("mechanical-switch")

for _, row in df_mechswitch.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_mechswitch.columns:
        if col in ["workorder_id", "filename"]:
            continue

        new_key = normalize_mechswitch_key(col)
        if new_key is not None:
            value = row[col]
            data[new_key] = to_bool(value, workorder_no, filename, col)
            continue

        data[col] = row[col]

    if workorder_no not in final_json:
        final_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        final_json[workorder_no]["data"].update(data)

        if not final_json[workorder_no].get("filename"):
            final_json[workorder_no]["filename"] = filename


In [158]:
bool_errors

[{'workorder_no': 4000673670,
  'filename': 'TN_PM_MTH_MechanicalSwitch1_4000673670.pdf',
  'column': 'drive_assembly.g.completed?',
  'value': 'Yés',
  'error': 'Unknown boolean format, storing as False'},
 {'workorder_no': 4000638202,
  'filename': 'TN_PM_MTH_MechanicalSwitch6_4000638202.pdf',
  'column': 'switch_deck_and_RC_beam.e.completed?',
  'value': 'Yés',
  'error': 'Unknown boolean format, storing as False'}]

### Finalize JSON

In [161]:
CUSTOM_PREFIX_ORDER = [
    "switch_deck_and_rc_beam",
    "carriage_including_carriage_wheel",
    "locking_pin_assembly",
    "non_rumba_mechanism",
    "switch_beam",
    "drive_assembly",
    "faults", "any_additional_work_that_requires_planning",
    "technican", "supervisor"
]

prefix_rank = {p: i for i, p in enumerate(CUSTOM_PREFIX_ORDER)}

def sort_key(item):
    key = item[0]

    if "." in key:
        prefix, rest = key.split(".", 1)
    else:
        prefix, rest = key, ""

    if rest and "." in rest:
        letter = rest.split(".", 1)[0]
    elif rest:
        letter = rest
    else:
        letter = ""

    return (
        prefix_rank.get(prefix, 999),
        letter
    )


for wo, payload in final_json.items():
    sorted_items = sorted(
        payload["data"].items(),
        key=sort_key
    )
    payload["data"] = dict(sorted_items)

final_json

{4000484591: {'filename': 'TN_PM_MTH_MechanicalSwitch1_4000484591.pdf',
  'data': {'switch_deck_and_rc_beam.a.inspection_item': 'Locking pin recesses drained of water.',
   'switch_deck_and_rc_beam.a.status': True,
   'switch_deck_and_rc_beam.b.inspection_item': 'Locking pin guide plate clearance 1mm maximum between cam roller and plates.',
   'switch_deck_and_rc_beam.b.status': True,
   'switch_deck_and_rc_beam.c.inspection_item': 'Carriage rail retaining bolts tight and clamping rail to plinth.',
   'switch_deck_and_rc_beam.c.status': True,
   'switch_deck_and_rc_beam.d.inspection_item': 'Switch deck drainage clean.',
   'switch_deck_and_rc_beam.d.status': True,
   'switch_deck_and_rc_beam.e.inspection_item': 'Switch deck auxiliaries fastened to the switch deck.',
   'switch_deck_and_rc_beam.e.status': True,
   'switch_deck_and_rc_beam.f.inspection_item': 'Finger plates bolted securely.',
   'switch_deck_and_rc_beam.f.status': True,
   'switch_deck_and_rc_beam.g.inspection_item': 'Li

In [ ]:
import pandas as pd
import json

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/responses_tnm_mechswitch.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")


Saved as: ../../output/responses_tnm_mechswitch.xlsx


In [ ]:
process_mechswitch_sheet(dfs.get("hitachi"), "hitachi")
process_mechswitch_sheet(dfs.get("mhe-demag"), "mhe-demag")
process_mechswitch_sheet(dfs.get("srb"), "srb")
process_mechswitch_sheet(dfs.get("mechanical-switch"), "mechanical-switch")

In [ ]:
df_ref = pd.DataFrame([
    {
        "key": k,
        "inspection_item": v["inspection_item"],
        "brand": ", ".join(sorted(v["brands"]))
    }
    for k, v in mechswitch_reference.items()
])

with pd.ExcelWriter(
    "../../output/responses_tnm_mechswitch.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_ref.to_excel(writer, sheet_name="mechswitch_reference", index=False)


In [ ]:
df_ref

,key,inspection_item,brand
0,non_rumba_mechanism.e.status,Non-Rumba plates secured tightly at each end.,hitachi
1,non_rumba_mechanism.f.status,Non-Rumba cams firmly secured.,hitachi
2,non_rumba_mechanism.g.status,Remove manhole cover and inspect internally th...,hitachi
3,switch_beam.a.status,Top and side finger plates secured.,hitachi
4,switch_beam.b.status,Beam and carriage securely fastened together.,hitachi
5,switch_beam.c.status,Remove tee shaft covers and inspect bushes for...,hitachi
6,switch_beam.d.status,Remove pivot point end cap and inspect.,hitachi
7,switch_beam.e.status,Cam guide boxes securely fastened to beam unde...,hitachi
8,switch_beam.f.status,Clearance between cam roller and guide box is ...,hitachi
9,drive_assembly.g.status,Check all gearboxes are securely fastened to m...,hitachi
